# Ridge And Lasso Logistic Regression

This notebook trains and compares ridge logistic regression and lasso logistic regression on the Tommy Award same-game dataset.

It uses the top same-game feature set from `Same_Game_And_Pregame_Logistic_Experiments.ipynb` so the notebook matches the feature style you have already been using.

In [1]:
# Import the libraries used for loading data, preprocessing,
# model training, evaluation, and coefficient inspection.
import warnings
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*penalty.*deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*penalty=l1.*", category=UserWarning)

# Basic configuration shared across the notebook.
INPUT_PATH = "Tommy_Award_Player_Game_Table.csv"
RANDOM_STATE = 42
MIN_TRAIN_SEASONS = 3
PRED_THRESHOLD = 0.5

In [2]:
# Load the dataset, recreate any missing derived same-game features,
# and define the shared feature list from Same_Game_And_Pregame_Logistic_Experiments.ipynb.
def load_dataset(path: str = INPUT_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"GAME_ID": str}).copy()
    df["game_date"] = pd.to_datetime(df["game_date"], format="mixed")

    # Keep only players who actually played in the game.
    df = df[df["minutes_decimal"] > 0].copy()

    mins = df["minutes_decimal"].clip(lower=1e-6)
    if "stocks" not in df.columns:
        df["stocks"] = df["steals"] + df["blocks"]
    if "points_per_min" not in df.columns:
        df["points_per_min"] = df["points"] / mins
    if "oreb_per_min" not in df.columns:
        df["oreb_per_min"] = df["reboundsOffensive"] / mins
    if "reb_per_min" not in df.columns:
        df["reb_per_min"] = df["reboundsTotal"] / mins
    if "ast_per_min" not in df.columns:
        df["ast_per_min"] = df["assists"] / mins
    if "hustle_proxy" not in df.columns:
        df["hustle_proxy"] = df["reboundsOffensive"] + df["steals"] + df["blocks"]
    if "stocks_per_min" not in df.columns:
        df["stocks_per_min"] = df["stocks"] / mins

    rank_sources = {
        "points_rank": "points",
        "reboundsOffensive_rank": "reboundsOffensive",
        "reboundsTotal_rank": "reboundsTotal",
        "assists_rank": "assists",
        "steals_rank": "steals",
        "blocks_rank": "blocks",
        "plusMinusPoints_rank": "plusMinusPoints",
        "minutes_decimal_rank": "minutes_decimal",
        "stocks_rank": "stocks",
        "hustle_proxy_rank": "hustle_proxy",
    }
    for rank_col, source_col in rank_sources.items():
        if rank_col not in df.columns:
            df[rank_col] = df.groupby("GAME_ID")[source_col].rank(method="min", ascending=False)

    if "season" not in df.columns:
        start_year = df["game_date"].dt.year.where(df["game_date"].dt.month >= 10, df["game_date"].dt.year - 1)
        df["season"] = start_year.astype(str) + "-" + (start_year + 1).astype(str).str[-2:]

    return df


def sorted_seasons(seasons: list[str]) -> list[str]:
    return sorted(seasons, key=lambda season: int(str(season).split("-")[0]))


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols = [
        "minutes_decimal",
        "points",
        "reboundsOffensive",
        "reboundsDefensive",
        "reboundsTotal",
        "assists",
        "steals",
        "blocks",
        "turnovers",
        "foulsPersonal",
        "plusMinusPoints",
        "fieldGoalsMade",
        "fieldGoalsAttempted",
        "threePointersMade",
        "threePointersAttempted",
        "freeThrowsMade",
        "stocks",
        "points_per_min",
        "oreb_per_min",
        "reb_per_min",
        "ast_per_min",
        "stocks_per_min",
        "hustle_proxy",
        "points_rank",
        "reboundsOffensive_rank",
        "reboundsTotal_rank",
        "assists_rank",
        "steals_rank",
        "blocks_rank",
        "plusMinusPoints_rank",
        "minutes_decimal_rank",
        "stocks_rank",
        "hustle_proxy_rank",
    ]

    missing_cols = [col for col in numeric_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols}")

    return numeric_cols, []


def walk_forward_season_splits(
    df: pd.DataFrame,
    min_train_seasons: int = MIN_TRAIN_SEASONS,
) -> list[tuple[list[str], str]]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    return [(seasons[:idx], seasons[idx]) for idx in range(min_train_seasons, len(seasons))]


def latest_season_holdout_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    train_df = df[df["season"].isin(seasons[:-1])].copy()
    test_df = df[df["season"] == seasons[-1]].copy()
    return train_df, test_df

In [3]:
# Build the numeric preprocessing and the two logistic regression models.
def build_preprocessor(numeric_cols: list[str], categorical_cols: list[str]) -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
        ]
    )


def build_ridge_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    return Pipeline(
        steps=[
            ("prep", build_preprocessor(numeric_cols, categorical_cols)),
            (
                "clf",
                LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    solver="lbfgs",
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def build_lasso_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    return Pipeline(
        steps=[
            ("prep", build_preprocessor(numeric_cols, categorical_cols)),
            (
                "clf",
                LogisticRegression(
                    penalty="l1",
                    C=1.0,
                    solver="liblinear",
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def available_model_names() -> list[str]:
    return ["ridge_logistic", "lasso_logistic"]

In [4]:
# Score predictions using the same game-level and row-level metrics
# used in your other modeling notebooks.
def score_predictions(scored_df: pd.DataFrame, threshold: float = PRED_THRESHOLD) -> dict[str, float]:
    pred_winners = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(1)
        .copy()
    )
    top3 = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(3)
        .copy()
    )

    y_true = scored_df["y"]
    y_prob = scored_df["pred_prob"].clip(1e-6, 1 - 1e-6)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "game_level_accuracy": pred_winners["y"].mean(),
        "top3_accuracy": top3.groupby("GAME_ID")["y"].max().mean(),
        "row_prauc": average_precision_score(y_true, y_prob),
        "row_recall": recall_score(y_true, y_pred, zero_division=0),
        "row_precision": precision_score(y_true, y_pred, zero_division=0),
        "row_logloss": log_loss(y_true, y_prob, labels=[0, 1]),
        "row_auc": roc_auc_score(y_true, y_prob),
    }


def fit_and_score_model(
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> tuple[Pipeline, pd.DataFrame, dict[str, float]]:
    feature_cols = numeric_cols + categorical_cols
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df["y"]

    if model_name == "ridge_logistic":
        model = build_ridge_pipeline(numeric_cols, categorical_cols)
    elif model_name == "lasso_logistic":
        model = build_lasso_pipeline(numeric_cols, categorical_cols)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    model.fit(X_train, y_train)

    scored_df = test_df.copy()
    scored_df["pred_prob"] = model.predict_proba(X_test)[:, 1]
    metrics = score_predictions(scored_df)
    return model, scored_df, metrics


def rank_results(results_df: pd.DataFrame) -> pd.DataFrame:
    ranked_df = results_df.sort_values(
        by=[
            "game_level_accuracy",
            "top3_accuracy",
            "row_prauc",
            "row_recall",
            "row_precision",
            "row_auc",
            "row_logloss",
        ],
        ascending=[False, False, False, False, False, False, True],
    ).reset_index(drop=True)
    ranked_df.insert(0, "rank", range(1, len(ranked_df) + 1))
    return ranked_df

In [5]:
# Load the data, inspect the shared feature set, and summarize the modeling setup.
df = load_dataset()
numeric_cols, categorical_cols = get_feature_columns(df)
model_names = available_model_names()
train_df, latest_test_df = latest_season_holdout_split(df)
latest_season = sorted_seasons(df["season"].dropna().unique().tolist())[-1]

summary_df = pd.DataFrame(
    {
        "item": [
            "rows",
            "numeric_features",
            "categorical_features",
            "train_games",
            "test_games",
            "train_positive_rate",
            "test_season",
            "models",
        ],
        "value": [
            len(df),
            len(numeric_cols),
            len(categorical_cols),
            train_df["GAME_ID"].nunique(),
            latest_test_df["GAME_ID"].nunique(),
            round(train_df["y"].mean(), 4),
            latest_season,
            ", ".join(model_names),
        ],
    }
)

display(summary_df)

,item,value
0,rows,8377
1,numeric_features,33
2,categorical_features,0
3,train_games,707
4,test_games,71
5,train_positive_rate,0.0929
6,test_season,2025-26
7,models,"ridge_logistic, lasso_logistic"


In [6]:
# Walk-forward evaluation tests each future season using only earlier seasons for training.
walk_forward_rows = []

for train_seasons, test_season in walk_forward_season_splits(df):
    split_train_df = df[df["season"].isin(train_seasons)].copy()
    split_test_df = df[df["season"] == test_season].copy()

    for model_name in model_names:
        _, _, metrics = fit_and_score_model(
            model_name,
            split_train_df,
            split_test_df,
            numeric_cols,
            categorical_cols,
        )
        walk_forward_rows.append(
            {
                "model": model_name,
                "test_season": test_season,
                **metrics,
            }
        )

walk_forward_df = pd.DataFrame(walk_forward_rows)
walk_forward_summary_df = (
    walk_forward_df.groupby("model", as_index=False)[
        [
            "game_level_accuracy",
            "top3_accuracy",
            "row_prauc",
            "row_recall",
            "row_precision",
            "row_logloss",
            "row_auc",
        ]
    ]
    .mean()
)
walk_forward_ranked_df = rank_results(walk_forward_summary_df)

display(walk_forward_df.round(4))
display(walk_forward_ranked_df.round(4))

,model,test_season,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_logloss,row_auc
0,ridge_logistic,2019-20,0.4861,0.7917,0.4659,0.7778,0.2772,0.4200,0.8855
1,lasso_logistic,2019-20,0.5000,0.7917,0.4662,0.7639,0.2709,0.4198,0.8863
2,ridge_logistic,2020-21,0.2083,0.7083,0.2267,0.7361,0.2524,0.5129,0.8113
3,lasso_logistic,2020-21,0.2083,0.7083,0.2268,0.7361,0.2512,0.5121,0.8113
4,ridge_logistic,2021-22,0.3500,0.7500,0.3145,0.8000,0.2530,0.4959,0.8451
5,lasso_logistic,2021-22,0.3500,0.7500,0.3138,0.7875,0.2500,0.4963,0.8450
6,ridge_logistic,2022-23,0.2195,0.6829,0.2347,0.7317,0.2143,0.5714,0.7921
7,lasso_logistic,2022-23,0.2195,0.6829,0.2347,0.7317,0.2135,0.5719,0.7920
8,ridge_logistic,2023-24,0.3165,0.7089,0.3157,0.8354,0.2157,0.5887,0.8257
9,lasso_logistic,2023-24,0.3165,0.7089,0.3155,0.8354,0.2157,0.5886,0.8258


,rank,model,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_logloss,row_auc
0,1,lasso_logistic,0.3251,0.7369,0.3238,0.7893,0.2396,0.5188,0.8365
1,2,ridge_logistic,0.3231,0.7352,0.3238,0.7931,0.2410,0.5188,0.8363


In [7]:
# Train each model on all earlier seasons and evaluate on the newest season only.
latest_rows = []
trained_models = {}
latest_scored_outputs = {}

for model_name in model_names:
    model, scored_df, metrics = fit_and_score_model(
        model_name,
        train_df,
        latest_test_df,
        numeric_cols,
        categorical_cols,
    )
    trained_models[model_name] = model
    latest_scored_outputs[model_name] = scored_df
    latest_rows.append({"model": model_name, "test_season": latest_season, **metrics})

latest_results_df = pd.DataFrame(latest_rows)
latest_ranked_df = rank_results(latest_results_df)

display(latest_ranked_df.round(4))

,rank,model,test_season,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_logloss,row_auc
0,1,lasso_logistic,2025-26,0.3521,0.7606,0.3703,0.8169,0.2407,0.4950,0.8560
1,2,ridge_logistic,2025-26,0.3521,0.7606,0.3699,0.8169,0.2397,0.4954,0.8557


In [8]:
# Inspect the largest coefficients from each trained model.
# Ridge keeps all coefficients, while lasso may push some exactly to zero.
coefficient_tables = {}

for model_name, pipeline in trained_models.items():
    log_model = pipeline.named_steps["clf"]
    coef_df = pd.DataFrame(
        {
            "feature": numeric_cols,
            "coefficient": log_model.coef_[0],
        }
    )
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
    coefficient_tables[model_name] = coef_df

    print(f"\nTop coefficients: {model_name}")
    display(coef_df.head(20).round(4))


Top coefficients: ridge_logistic


,feature,coefficient,abs_coefficient
12,fieldGoalsAttempted,-1.6360,1.6360
23,points_rank,-1.2127,1.2127
11,fieldGoalsMade,0.8567,0.8567
1,points,0.5397,0.5397
0,minutes_decimal,0.5080,0.5080
19,reb_per_min,0.4621,0.4621
29,plusMinusPoints_rank,-0.4297,0.4297
17,points_per_min,0.4192,0.4192
5,assists,0.3232,0.3232
14,threePointersAttempted,0.3216,0.3216



Top coefficients: lasso_logistic


,feature,coefficient,abs_coefficient
12,fieldGoalsAttempted,-1.6279,1.6279
11,fieldGoalsMade,1.2465,1.2465
23,points_rank,-1.2121,1.2121
0,minutes_decimal,0.4952,0.4952
19,reb_per_min,0.4451,0.4451
29,plusMinusPoints_rank,-0.4278,0.4278
17,points_per_min,0.4117,0.4117
22,hustle_proxy,0.3512,0.3512
14,threePointersAttempted,0.3139,0.3139
5,assists,0.3077,0.3077


In [9]:
# Show the full shared feature set used by both models.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print("Numeric features")
display(pd.DataFrame({"numeric_feature": numeric_cols}))

print("Categorical features")
display(pd.DataFrame({"categorical_feature": categorical_cols}))

Numeric features


,numeric_feature
0,minutes_decimal
1,points
2,reboundsOffensive
3,reboundsDefensive
4,reboundsTotal
5,assists
6,steals
7,blocks
8,turnovers
9,foulsPersonal


Categorical features


,categorical_feature
